In [15]:
# ============================================================
# IN-CONTEXT LEARNING EXPERIMENT (FINAL CORRECTED VERSION)
#
# Model: google/flan-t5-small (Instruction-Tuned, Lightweight)
#
# Prompting Strategies:
#   - Zero-shot
#   - One-shot
#   - Few-shot
#   - Chain-of-Thought (CoT)
#
# Fixes Implemented:
#   ✔ Dataset shuffled (removes class imbalance bias)
#   ✔ Balanced sampling
#   ✔ Proper evaluation metrics
#   ✔ Clean label extraction
#
# Metrics:
#   - Accuracy
#   - Precision
#   - Recall
#   - F1-score
#   - Confusion Matrix
# ============================================================



# ============================================================
# STEP 1: IMPORT LIBRARIES
# ============================================================

# HuggingFace model and tokenizer
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# Dataset loader
from datasets import load_dataset

# PyTorch backend
import torch

# Progress bar
from tqdm import tqdm

# Numerical library
import numpy as np

# Classification metrics
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix



# ============================================================
# STEP 2: LOAD INSTRUCTION-TUNED MODEL
# ============================================================

# We use FLAN-T5-small:
# - Lightweight (~80M parameters)
# - Instruction-tuned
# - Fast on CPU
model_name = "google/flan-t5-small"

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Load model (encoder-decoder architecture)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

# Detect device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Move model to device
model.to(device)

# Disable dropout for stable evaluation
model.eval()



# ============================================================
# STEP 3: LOAD AND BALANCE DATASET
# ============================================================

# Load full test split
dataset = load_dataset("imdb", split="test")

# Shuffle dataset to avoid order bias
dataset = dataset.shuffle(seed=42)

# Select first 100 shuffled samples
dataset = dataset.select(range(100))

# Extract review texts
inputs = dataset["text"]

# Convert numeric labels to string labels
true_labels = np.array(
    ["Positive" if label == 1 else "Negative"
     for label in dataset["label"]]
)

# Display class distribution (important for debugging imbalance)
print("\nClass Distribution:")
print(np.unique(true_labels, return_counts=True))



# ============================================================
# STEP 4: DEFINE PROMPTING STRATEGIES
# ============================================================

# ZERO-SHOT: No examples provided
def zero_shot_prompt(text):
    return f"""
Classify the sentiment of this movie review as Positive or Negative.
Answer with only one word.

Review: {text}
Answer:
"""


# ONE-SHOT: One labeled example provided
def one_shot_prompt(text):
    return f"""
Classify the sentiment.

Review: I loved this movie. It was fantastic.
Answer: Positive

Review: {text}
Answer:
"""


# FEW-SHOT: Multiple labeled examples
def few_shot_prompt(text):
    return f"""
Classify the sentiment.

Review: I loved this movie.
Answer: Positive

Review: This movie was terrible.
Answer: Negative

Review: Amazing acting and great story.
Answer: Positive

Review: {text}
Answer:
"""


# CHAIN-OF-THOUGHT (CoT): Add reasoning step
def cot_prompt(text):
    return f"""
Classify the sentiment step by step.

Review: I loved this movie.
Reasoning: The phrase shows strong positive emotion.
Answer: Positive

Review: {text}
Reasoning:
"""



# ============================================================
# STEP 5: GENERATION FUNCTION
# ============================================================

def generate_answer(prompt):
    """
    Converts prompt to tokens,
    generates model output,
    converts output tokens back to text.
    """

    # Tokenize input prompt
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=512
    ).to(device)

    # Generate output deterministically
    outputs = model.generate(
        **inputs,
        max_new_tokens=15
    )

    # Decode tokens into readable text
    decoded_output = tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

    return decoded_output.strip()



# ============================================================
# STEP 6: EVALUATION FUNCTION
# ============================================================

def evaluate_prompting(strategy_function, name):

    predictions = []

    print(f"\nEvaluating {name}...\n")

    for text in tqdm(inputs):

        # Limit review length for speed
        prompt = strategy_function(text[:300])

        # Generate model output
        output = generate_answer(prompt).lower()

        # Extract label safely
        if "positive" in output:
            predictions.append("Positive")
        elif "negative" in output:
            predictions.append("Negative")
        else:
            # If unclear, assign majority class prediction
            predictions.append("Negative")

    predictions = np.array(predictions)

    # Convert to binary for metrics
    y_true = np.array([1 if x == "Positive" else 0 for x in true_labels])
    y_pred = np.array([1 if x == "Positive" else 0 for x in predictions])

    # Compute evaluation metrics
    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, zero_division=0)
    rec = recall_score(y_true, y_pred, zero_division=0)
    f1 = f1_score(y_true, y_pred, zero_division=0)
    cm = confusion_matrix(y_true, y_pred)

    print(f"\n{name} Results:")
    print(f"Accuracy:  {acc:.4f}")
    print(f"Precision: {prec:.4f}")
    print(f"Recall:    {rec:.4f}")
    print(f"F1-score:  {f1:.4f}")
    print("Confusion Matrix:\n", cm)

    return acc, f1



# ============================================================
# STEP 7: RUN ALL PROMPT STRATEGIES
# ============================================================

zero_acc, zero_f1 = evaluate_prompting(zero_shot_prompt, "Zero-Shot")
one_acc, one_f1 = evaluate_prompting(one_shot_prompt, "One-Shot")
few_acc, few_f1 = evaluate_prompting(few_shot_prompt, "Few-Shot")
cot_acc, cot_f1 = evaluate_prompting(cot_prompt, "Chain-of-Thought")



# ============================================================
# FINAL SUMMARY
# ============================================================

print("\n============================")
print("FINAL SUMMARY")
print("============================")

print(f"Zero-Shot  → Acc: {zero_acc:.4f}, F1: {zero_f1:.4f}")
print(f"One-Shot   → Acc: {one_acc:.4f}, F1: {one_f1:.4f}")
print(f"Few-Shot   → Acc: {few_acc:.4f}, F1: {few_f1:.4f}")
print(f"CoT        → Acc: {cot_acc:.4f}, F1: {cot_f1:.4f}")


C:\Users\qawsar.gulzar\AppData\Local\anaconda3\Lib\site-packages\huggingface_hub\file_download.py:942: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(



Class Distribution:
(array(['Negative', 'Positive'], dtype='<U8'), array([53, 47], dtype=int64))

Evaluating Zero-Shot...



100%|████████████████████████████████████████████████████████████████████████████████| 100/100 [00:10<00:00,  9.74it/s]



Zero-Shot Results:
Accuracy:  0.8500
Precision: 0.9000
Recall:    0.7660
F1-score:  0.8276
Confusion Matrix:
 [[49  4]
 [11 36]]

Evaluating One-Shot...



100%|████████████████████████████████████████████████████████████████████████████████| 100/100 [00:08<00:00, 11.50it/s]



One-Shot Results:
Accuracy:  0.8400
Precision: 0.8974
Recall:    0.7447
F1-score:  0.8140
Confusion Matrix:
 [[49  4]
 [12 35]]

Evaluating Few-Shot...



100%|████████████████████████████████████████████████████████████████████████████████| 100/100 [00:09<00:00, 10.18it/s]



Few-Shot Results:
Accuracy:  0.8300
Precision: 0.8750
Recall:    0.7447
F1-score:  0.8046
Confusion Matrix:
 [[48  5]
 [12 35]]

Evaluating Chain-of-Thought...



100%|████████████████████████████████████████████████████████████████████████████████| 100/100 [00:15<00:00,  6.64it/s]


Chain-of-Thought Results:
Accuracy:  0.8500
Precision: 0.8636
Recall:    0.8085
F1-score:  0.8352
Confusion Matrix:
 [[47  6]
 [ 9 38]]

FINAL SUMMARY
Zero-Shot  → Acc: 0.8500, F1: 0.8276
One-Shot   → Acc: 0.8400, F1: 0.8140
Few-Shot   → Acc: 0.8300, F1: 0.8046
CoT        → Acc: 0.8500, F1: 0.8352
